# Fake News Detection System using Machine Learning & NLP
**Author:** AI/ML Fresher Candidate  
**Degree Context:** B.Tech Artificial Intelligence & Data Science  

--- 
## Notebook Overview
This notebook demonstrates an end-to-end Machine Learning and Natural Language Processing (NLP) pipeline for detecting fake news articles.  
Key Pipeline Steps:
1. **Exploratory Data Analysis (EDA)** & Visualization
2. **Text Preprocessing** (Lowercasing, Punctuation Removal, Stopwords Removal, Porter Stemming)
3. **Feature Engineering** using TF-IDF Vectorizer
4. **Model Building & Evaluation** comparing Logistic Regression, Naive Bayes, and Random Forest
5. **Model Export** for Streamlit Web Application deployment

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

# Set plot styles
plt.style.use('ggplot')
sns.set_palette('muted')

## 1. Load and Clean Dataset

In [ ]:
df_fake = pd.read_csv('../dataset/Fake.csv')
df_true = pd.read_csv('../dataset/True.csv')

df_fake['label'] = 0
df_true['label'] = 1

df = pd.concat([df_fake, df_true], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
df['combined_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
print('Dataset Shape:', df.shape)
df.head()

## 2. Dataset Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, palette=['#dc3545', '#28a745'])
plt.xticks([0, 1], ['Fake (0)', 'Real (1)'])
plt.title('Distribution of Fake vs Real News Articles')
plt.show()

## 3. Text Preprocessing & NLP Pipeline

In [ ]:
import sys
sys.path.append('../')
from preprocessing import preprocess_text

df['processed_text'] = df['combined_text'].apply(preprocess_text)
df[['combined_text', 'processed_text']].head()

## 4. Word Cloud Visualization

In [ ]:
fake_words = ' '.join(df[df['label']==0]['processed_text'])
real_words = ' '.join(df[df['label']==1]['processed_text'])

wc_fake = WordCloud(width=800, height=400, background_color='black', colormap='Reds').generate(fake_words)
wc_real = WordCloud(width=800, height=400, background_color='black', colormap='Greens').generate(real_words)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(wc_fake, interpolation='bilinear')
ax[0].set_title('Most Frequent Words in FAKE News')
ax[0].axis('off')

ax[1].imshow(wc_real, interpolation='bilinear')
ax[1].set_title('Most Frequent Words in REAL News')
ax[1].axis('off')

plt.tight_layout()
plt.show()

## 5. TF-IDF Feature Extraction & Model Evaluation

In [ ]:
vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vec.fit_transform(df['processed_text'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f'=== {name} ===')
    print(classification_report(y_test, preds))
